<a href="https://colab.research.google.com/github/asmaatefomran/generative-ai-tasks/blob/main/Task3_fine_tuning_open_source_llm_medical_conversations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 3: Fine-Tuning an Open-Source LLM for Medical Conversations**

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


GPU available: True
GPU: Tesla T4


Install packages

In [2]:
!pip install -q transformers datasets peft accelerate torchao


Imports

In [3]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model


Load the small model

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)

model = model.to("cuda")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
tokenizer.pad_token = tokenizer.eos_token


Small instruction dataset

In [6]:
data = [
    {
        "instruction": "What is a fever?",
        "response": "A fever is a temporary increase in body temperature, often caused by an infection."
    },
    {
        "instruction": "What are common symptoms of the flu?",
        "response": "Common flu symptoms include fever, cough, sore throat, body aches, fatigue, and headache."
    },
    {
        "instruction": "What is dehydration?",
        "response": "Dehydration occurs when the body loses more fluids than it takes in. It can cause thirst, dry mouth, dizziness, and reduced urination."
    },
    {
        "instruction": "Why is drinking water important?",
        "response": "Water helps maintain hydration, regulate body temperature, transport nutrients, and support normal body functions."
    },
    {
        "instruction": "What is a headache?",
        "response": "A headache is pain or discomfort occurring in the head or upper neck area."
    },
    {
        "instruction": "What are common symptoms of a cold?",
        "response": "Common cold symptoms include a runny nose, congestion, sneezing, sore throat, cough, and mild fatigue."
    },
    {
        "instruction": "What is blood pressure?",
        "response": "Blood pressure is the force of blood pushing against the walls of the arteries as the heart pumps blood."
    },
    {
        "instruction": "Why is sleep important?",
        "response": "Sleep helps the body and brain recover, supports memory, regulates mood, and contributes to overall health."
    }
]

dataset = Dataset.from_list(data)

print(dataset)


Dataset({
    features: ['instruction', 'response'],
    num_rows: 8
})


Format the dataset

In [7]:
def format_text(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}"""
    }

dataset = dataset.map(format_text)

print(dataset[0]["text"])


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

### Instruction:
What is a fever?

### Response:
A fever is a temporary increase in body temperature, often caused by an infection.


Tokenize the dataset

In [8]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=False
)


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Configure LoRA

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

In [11]:
!pip install -q -U "peft>=0.18.0" "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 76.2 MB/s eta 0:00:00


In [12]:
model = get_peft_model(
    model,
    lora_config
)

In [13]:
model.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Train the model

In [14]:
training_args = TrainingArguments(
    output_dir="./medical_lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)


In [16]:
trainer.train()


Step,Training Loss
1,2.082219
2,1.791424


TrainOutput(global_step=2, training_loss=1.9368211030960083, metrics={'train_runtime': 3.1592, 'train_samples_per_second': 2.532, 'train_steps_per_second': 0.633, 'total_flos': 6362964688896.0, 'train_loss': 1.9368211030960083, 'epoch': 1.0})

Save the LoRA adapter

In [17]:
adapter_path = "./medical_lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully.")


LoRA adapter saved successfully.


Test the model BEFORE fine-tuning

In [18]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [19]:
def generate_response(model, question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.7,
            do_sample=True
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


In [20]:
question = "What is dehydration?"

before = generate_response(
    base_model,
    question
)

print(before)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is dehydration?

### Response:
Dehydration is a condition where the body does not have enough water to function properly. When the body loses water, it becomes thinner, which reduces the volume of the blood and the capillaries. This makes it harder for the body to transport nutrients and oxygen to various parts of the body. When dehydration occurs, it can lead to a


Test the fine-tuned model

In [21]:
after = generate_response(
    model,
    question
)

print(after)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is dehydration?

### Response:
Dehydration is a condition where the body loses water and fluids. It occurs when the body stops producing more sweat, lost fluids through urine, and reduced water intake. This condition can lead to fatigue, weakness, headaches, confusion, and other symptoms. Dehydration can be prevented by drinking enough water, staying


Compare Before vs After

In [22]:
print("=" * 60)
print("BEFORE FINE-TUNING")
print("=" * 60)

print(before)

print("\n")

print("=" * 60)
print("AFTER FINE-TUNING")
print("=" * 60)

print(after)


BEFORE FINE-TUNING
### Instruction:
What is dehydration?

### Response:
Dehydration is a condition where the body does not have enough water to function properly. When the body loses water, it becomes thinner, which reduces the volume of the blood and the capillaries. This makes it harder for the body to transport nutrients and oxygen to various parts of the body. When dehydration occurs, it can lead to a


AFTER FINE-TUNING
### Instruction:
What is dehydration?

### Response:
Dehydration is a condition where the body loses water and fluids. It occurs when the body stops producing more sweat, lost fluids through urine, and reduced water intake. This condition can lead to fatigue, weakness, headaches, confusion, and other symptoms. Dehydration can be prevented by drinking enough water, staying


In [23]:
questions = [
    "What is a fever?",
    "What are symptoms of the flu?",
    "Why is sleep important?",
    "What is blood pressure?"
]

for question in questions:

    print("\n" + "=" * 60)
    print("QUESTION:", question)

    print("\nBEFORE:")
    print(generate_response(base_model, question))

    print("\nAFTER:")
    print(generate_response(model, question))


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is a fever?

BEFORE:


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is a fever?

### Response:
A fever is a temperature above 100.4 degrees Fahrenheit, also known as a high-temperature, and it is a sign of a serious medical condition. It can be caused by infection, inflammation, or a medical condition. A fever is a sign that your body is fighting infection or inflammation, and it can help diagnose and

AFTER:


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

### Instruction:
Why is sleep important?

### Response:
Sleep is important because it is essential for maintaining energy levels, mental clarity, and physical health. Sleep helps regulate our body's internal clock and helps us wake up feeling refreshed and ready to tackle the day ahead. It also helps us store energy in our fat cells, which can be used during times of stress or lack of food. Sleep is also important

AFTER:


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why is sleep important?

### Response:
Sleep is essential for maintaining physical and mental health and functioning. It helps in regulating the body's internal clock, maintaining a healthy immune system, and improving cognitive function. Sleep enables us to remember and learn new things, and it helps us feel more alert and energy-filled throughout the day. Additionally, sleep is important for repairing and restoring

QUESTION: What is blood pressure?

BEFORE:


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
What is blood pressure?

### Response:
Blood pressure is the force of blood pushing back against the walls of the arteries in your body. It is measured as a pressure of 120 mm Hg or less. High blood pressure, also known as hypertension, is a serious condition that can cause damage to your arteries and increase your risk of heart disease and stroke. Hypertension can be caused

AFTER:
### Instruction:
What is blood pressure?

### Response:
Blood pressure refers to the pressure of blood flowing in or out of the blood vessels. It is measured in millimeters of mercury (mmHg) and is typically expressed as a number, such as "120/80". High blood pressure, also known as hypertension, is a condition where blood pressure is higher than normal. When blood pressure is
